# LangChain 01 · 模型 · 消息 · 结构化输出

这是 `02_langchain/` 的第一课。目标是把「跟大模型对话」这件事的**三个底座**一次讲清：

| 概念 | 是什么 | 本节的代码形态 |
|---|---|---|
| 模型 Model | 统一封装各家大模型的入口 | `init_chat_model(...)` / `ChatOpenAI(...)` |
| 消息 Message | 对话的底层单位（4 种角色） | `SystemMessage` / `HumanMessage` / `AIMessage` / `ToolMessage` |
| 结构化输出 | 让模型按 Pydantic schema 吐出 JSON，而不是自由文本 | `llm.with_structured_output(...)` / `create_agent(response_format=...)` |
| 模型配置进阶 | 温度 / 超时 / 重试 / 限流 / token 用量核算 | `temperature` / `max_retries` / `rate_limiter` / `usage_metadata` |

前三项是「把模型接上、把话说清楚、把答案规整化」，最后一项是「接上之后的工程细节」。

> **本 notebook 由 `Agent/02_langchain/` 下 7 个脚本合并而成**：
> `01_模型.py`（原版 47 行）+ `01_模型_jxsd.py`（完整版 117 行）讲模型；
> `02_消息.py`（原版 52 行）+ `02_消息_jxsd.py`（完整版 151 行）讲消息；
> `08_结构化输出.py`（原版 74 行）+ `08_结构化输出_jxsd.py`（完整版 162 行）讲结构化输出；
> `23_模型配置进阶_官方补充.py`（316 行）讲配置进阶（官方文档补充篇）。

**官方文档**
- 模型（Models）：<https://docs.langchain.com/oss/python/langchain/models>
- 消息（Messages）：<https://docs.langchain.com/oss/python/langchain/messages>
- 结构化输出（Structured Output）：<https://docs.langchain.com/oss/python/langchain/structured-output>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 会真实调用 `.env` 里配置的大模型（本机 `deepseek-flash`） |
| 依赖 | `langchain` / `langchain-core` / `langchain-openai` / `pydantic`（本项目 venv 已装） |
| 密钥 | `settings.api_key`（仓库根 `.env` 已配置） |
| 前置服务 | 无（不起服务、不读数据库、不写磁盘） |
| 预计耗时 | 约 1~3 分钟（十几轮真实模型往返；结构化输出走降级快速失败） |

⚠️ **本机特有的一处降级，先交代清楚**：

第 4 节「结构化输出」依赖**端点的结构化输出能力**。本机 `.env` 指向 DeepSeek 的
`deepseek-flash`（**思考模型**），两条结构化输出的路都**不支持**，实测事实如下：

| 走法 | 本机结果 |
|---|---|
| 原生 `json_schema`（`response_format`） | `400 This response_format type is unavailable now` |
| 强制 `tool_choice`（function calling） | `400 Thinking mode does not support this tool_choice` |
| `method="json_mode"` | 能通 HTTP，但**不约束字段名**（实测把 `goal/steps` 输出成 `目标/步骤`） |

源文件里已经内置了「打印中文提示并跳过」的降级分支，本节原样保留。
所以**在本机看到的是那条 `[跳过]` 提示**（属于设计好的路径，不是报错）；
**换一个支持 `json_schema` 或 function calling 的端点（如 GPT-4o 系 / 非思考模型），
同一段代码即可跑通**，无需改代码。

## 本节地图

一条「模型 → 消息 → 结构化输出 → 配置进阶」的主线，对应七个源文件的合并关系：

```mermaid
graph LR
    A["模型<br/>init_chat_model / ChatOpenAI"] --> B["消息<br/>System / Human / AI / Tool"]
    A --> C["结构化输出<br/>Pydantic → JSON Schema"]
    A --> D["配置进阶<br/>重试 / 限流 / 用量"]
    B --> E["多轮对话<br/>append 消息"]
    C --> F{"端点支持<br/>json_schema / tool_choice?"}
    F -->|"否（本机）"| G["降级：打印提示并跳过"]
    F -->|"是"| H["直接得到 Pydantic 对象"]
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 节 | 讲什么 | 合并自 |
|---|---|---|
| 1. 模型（原版） | `init_chat_model` 最短接法 + 一句话调用 | `01_模型.py` |
| 2. 模型（完整版） | `ChatOpenAI` 直接 new vs `init_chat_model` 工厂；无状态验证 | `01_模型_jxsd.py` |
| 3. 消息 | 4 种角色 + 多轮 append + ToolMessage + 三种等价写法 | `02_消息.py` + `02_消息_jxsd.py` |
| 4. 结构化输出 | `with_structured_output` + `create_agent(response_format=...)`（本机降级） | `08_结构化输出.py` + `08_结构化输出_jxsd.py` |
| 5. 配置进阶 | 参数 / 重试 / 限流 / 用量 / 多模态的现实 | `23_模型配置进阶_官方补充.py` |

**与上下节的衔接**：

- 上一章 `01_langgraph/` 讲「把智能体看成一张图」，但图里的模型长什么样还没展开；
- 本节补上「模型怎么接、消息怎么拼、输出怎么规整」这三块砖；
- 下一课 `02_智能体与工具.ipynb` 会看到：消息列表里那个 `ToolMessage` 是怎么被
  `create_agent` 自动填进去的 —— 本节第 3 节先把它手工摆出来看清楚。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

## 0.1 前置条件自检

这一格只做两件事，**不发任何网络请求**：

1. 逐个 `import` 本节要用的包，缺哪个就打印中文提示；
2. 检查 `settings.api_key` / `settings.base_url` 有没有读到；

然后把结论写进 `READY` —— 后面**真正调模型的代码都包在 `if READY:` 里**，
条件不满足时它们会整体跳过并打印跳过提示，而不是抛一堆 Traceback。

> 第 4 节结构化输出与第 5 节的部分 Demo 自带了 try/except 兜底，即使 `READY=False`
> 也不会崩，但统一用 `READY` 兜底能让「没配密钥」的机器上跑得更干净。

In [ ]:
# ---------- 前置条件自检：缺东西就打印中文提示，后续调模型的单元格整体跳过 ----------
import importlib

READY = True

for _pkg in ("langchain", "langchain_openai", "langchain_core", "pydantic"):
    try:
        importlib.import_module(_pkg)
    except ImportError as exc:                  # noqa: BLE001 —— 缺依赖只需要提示，不必中断
        READY = False
        print(f"[跳过] 缺少依赖 {_pkg}：{exc}")

from config import settings

if not settings.api_key:
    READY = False
    print("[跳过] 没读到 API Key：请在仓库根 .env 里配好 API_KEY / BASE_URL / MODEL_NAME")
if not settings.base_url:
    READY = False
    print("[跳过] 没读到 BASE_URL：请在仓库根 .env 里配好 API_KEY / BASE_URL / MODEL_NAME")

print(f"模型：{settings.model_name}    接口：{settings.base_url}")
print("前置条件自检：", "就绪" if READY else "条件不足（下面调模型的格都会打印 [跳过] 提示）")

### 预期输出

```text
模型：deepseek-flash    接口：https://api.deepseek.com
前置条件自检： 就绪
```

## 1. 课案原版：`init_chat_model` 最短接法

课案原版只有 47 行，讲一件事：**用 `init_chat_model` 统一接各家大模型**。
好处是换供应商只改参数，业务代码完全不变 —— 所有 OpenAI 兼容服务
（DeepSeek、通义、Kimi、vLLM……）都这样接。

### 1.1 初始化模型

三件套（`api_key` / `base_url` / `model_name`）统一从 `settings` 读，不硬编码。

In [ ]:
from langchain.chat_models import init_chat_model
from config import settings

# 初始化模型：所有 OpenAI 兼容服务（DeepSeek、通义、Kimi、vLLM……）都这样接
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
    temperature=0.7,  # 采样温度，越高越随机
)

### 1.2 最简单调用 + 带角色调用

`invoke` 既能吃纯字符串（自动包成一条 `HumanMessage`），也能吃 `(角色, 内容)` 元组列表。

In [ ]:
# ---------- 最简单调用：一句话进，一句话出 ----------
if READY:
    response = llm.invoke("用一句话解释什么是 Agent")
    print("普通回复：", response.content)

    # ---------- 带系统提示词（角色设定） ----------
    response = llm.invoke(
        [
            ("system", "你是一位毒舌程序员，回答简短刻薄但专业。"),
            ("user", "Python 好学吗？"),
        ]
    )
    print("带角色回复：", response.content)

### 预期输出

> ⚠️ 下面的正文是模型这次生成的，每次运行都不同；稳定的是「两次调用都只返回
> `response.content` 这一行文本」这个结构。

```text
普通回复： Agent 是一个能感知环境、自主决策并采取行动、根据反馈迭代以完成特定目标的智能实体。
带角色回复： 好学。语法简单到像伪代码，三天能写 hello world，三周能写爬虫。

但“好学”不等于“学好”。框架、并发、性能、工程化、依赖地狱，照样把你按在地上摩擦。

结论：Python 入门容易，精通另说。连它都学不会，建议慎重考虑程序员这行。
```

两次调用同一条 `llm`，区别只在入参：前者是裸字符串，后者是 `[system, user]` 列表 ——
这就是「系统提示词」和「用户输入」在入参层面的分界。

## 2. 完整版：`ChatOpenAI` 直接 new vs `init_chat_model` 工厂

课案原文用 `ChatOpenAI` 直接 new 出模型对象；本项目统一走 `init_chat_model`。
两者返回值都能 `.invoke()`，**功能完全一样**，区别只在「可切换性」。

| 维度 | `ChatOpenAI(...)` 直接 new | `init_chat_model(...)` 工厂 |
|---|---|---|
| 代码依赖 | 强依赖 `langchain_openai` 这个包和类名 | 只依赖字符串 `"openai"`，不 import 供应商包 |
| 换供应商 | 要改 import + 改类名（通义/Claude 各不相同） | 只改 `model_provider` 一个字符串 |
| 配置驱动 | 模型名写死在代码里 | 可从 `.env` / 数据库读，天然支持多环境 |
| 私有参数 | 能用到供应商独有的参数（如 `logit_bias`） | 只能传通用参数，私有参数要走 `**kwargs` |
| 运行期类型 | `ChatOpenAI` | 仍然是 `ChatOpenAI`（运行时同一对象） |
| 适合场景 | 只用一家、要压榨该家私有能力 | 多供应商切换、配置化/平台化的项目 |

### 2.1 两种写法

`init_chat_model` 内部维护一张「供应商名 → 模型类」的注册表：它按 `model_provider`
找到 `langchain_openai.ChatOpenAI`，再把 `model` / `api_key` / `base_url` 当构造参数传进去。
换句话说：**`init_chat_model` 是「工厂」，`ChatOpenAI` 是「车间」**。

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI
from config import settings


# ---------- 1. 课案写法：ChatOpenAI 直接 new ----------
# 课案演示的就是这种写法，所以这里保留下来（CONVENTIONS 允许：
# 课案原文明确演示 ChatOpenAI 的小节可以用 ChatOpenAI，但参数必须来自 settings，不能硬编码）。
legacy_model = ChatOpenAI(
    api_key=settings.api_key,       # 课案的 setting.API_KEY → settings.api_key
    base_url=settings.base_url,     # 课案的 setting.BASE_URL → settings.base_url
    model=settings.model_name,      # 课案的 setting.MODEL_NAME → settings.model_name
    temperature=0,                  # 课案固定 0：要的是稳定、可复现的输出
)

# ---------- 2. 本项目统一写法：init_chat_model ----------
# 注意这里没有 import 任何供应商的模型类，只给了 model_provider="openai"。
# init_chat_model 内部维护了一张「供应商名 → 模型类」的注册表：
# 它按 model_provider 找到 langchain_openai.ChatOpenAI，
# 再把 model / api_key / base_url 当构造参数传进去。
# 换句话说：init_chat_model 是「工厂」，ChatOpenAI 是「车间」。
llm = init_chat_model(
    model_provider="openai",        # 供应商：openai / anthropic / google_genai ...
    model=settings.model_name,      # 模型名
    api_key=settings.api_key,       # OpenAI 兼容接口的三件套
    base_url=settings.base_url,
    temperature=0.7,                # 采样温度：越高越随机；课案这里是 0
)

### 2.2 类型对照：证明「运行时是同一个类」

这一步不调网络，只打印两个对象的真实类型。

In [ ]:
# ---------- 4. 打印两种写法的真实类型，证明「运行时是同一个类」 ----------
print("legacy_model 的实际类型：", type(legacy_model).__name__)
print("llm 的实际类型：         ", type(llm).__name__)

### 预期输出

```text
legacy_model 的实际类型： ChatOpenAI
llm 的实际类型：          ChatOpenAI
```

两个对象**运行时是同一个类** —— 工厂只是省掉了「记类名、import 供应商包」的体力活。

### 2.3 无状态验证：同一个模型对象可以反复 invoke

每次调用都是**独立的一次 HTTP 请求**，模型不会记得上一句 ——
「记忆」是我们在外面把历史消息拼进列表实现的（见第 3 节、以及后面的 05/06）。

In [ ]:
if READY:
    # ---------- 5. 课案原样调用：一句话进，一句话出 ----------
    # 模型对象统一用 invoke() 调用（异步版是 ainvoke）；传进去的字符串会被自动包成
    # 一条 HumanMessage —— 这就是课案里 model.invoke("你好") 能直接跑的原因。
    response = legacy_model.invoke("你好")
    print("\n[课案写法] response 对象：", type(response).__name__)
    print("[课案写法] response.content：", response.content)

    # ---------- 6. 工厂写法同样能调，并演示「消息列表」入参 ----------
    # invoke 既能吃纯字符串，也能吃 (角色, 内容) 元组列表或 Message 对象（见 02_消息_jxsd.py）
    response = llm.invoke(
        [
            ("system", "你是一位惜字如金的助手，回答不超过 20 个字。"),
            ("user", "用一句话解释什么是 Agent"),
        ]
    )
    print("\n[工厂写法] 带角色的回复：", response.content)

    # ---------- 7. 同一个模型对象可以反复 invoke，它是无状态的 ----------
    # 每次调用都是独立的一次 HTTP 请求，模型不会记得上一句，
    # 「记忆」是我们在外面把历史消息拼进列表实现的（见 05/06）。
    again = llm.invoke("我刚才问你的是什么？")
    print("[无状态验证] 再问一次：", again.content)

### 预期输出

> ⚠️ 下面三处 `response.content` 的正文是模型这次生成的，每次运行都不同；
> 稳定的是「第一行的 `response 对象： AIMessage`」这一句。

```text
[课案写法] response 对象： AIMessage
[课案写法] response.content： 你好！很高兴见到你 😊 有什么我可以帮你的吗？无论是问题解答、写作、翻译、编程还是其他事情，都可以告诉我。

[工厂写法] 带角色的回复： Agent是能感知环境并自主行动的智能体
[无状态验证] 再问一次： 你刚才问我的是：**“我刚才问你的是什么？”**

不过在这个对话里，我没有看到你在这之前发过其他消息。如果你是想让我回忆更早的提问，可能需要把前文再发一下，或者确认一下是否切断了上下文。
```

最后一行是重点：`again = llm.invoke("我刚才问你的是什么？")` 拿到的回复**不记得上一句** ——
因为这次请求只发了「我刚才问你的是什么？」这一条消息，模型没有状态，记忆全靠消息列表。

## 3. 消息：对话的底层单位

对话的底层单位是消息。LangChain 常用 4 种角色（下表来自 `02_消息_jxsd.py` 的完整版）：

| 顺序 | 类 | type 值 | 谁产生 | 作用 | 是否必需 |
|---|---|---|---|---|---|
| 1 | `SystemMessage` | system | 开发者 | 人设 / 规则 / 输出格式约束，放最前，通常一条 | 否（但强烈建议） |
| 2 | `HumanMessage` | human | 用户 | 本轮用户输入 | 是 |
| 3 | `AIMessage` | ai | 模型 | 模型回复；要调工具时里面带 `tool_calls` | 由模型产生 |
| 4 | `ToolMessage` | tool | 工具执行层 | 工具结果，必须用 `tool_call_id` 与请求配对 | 调工具时必需 |

**顺序铁律**：`system` 必须在最前（且只放最前），`human` / `ai` / `tool` 按时间顺序交替追加；
`ToolMessage` 必须紧跟在「发起该次 `tool_call` 的 `AIMessage`」之后，且 `tool_call_id`
要对得上，否则供应商接口直接报 400。

「消息列表 = 对话的完整上下文」——模型每次都是对着整个列表重新生成回复，
**「记忆」的本质就是不断把新消息 append 到列表里，再交给模型**。

### 3.1 课案原版：元组写法

原版用最简的 `(role, content)` 元组写法演示「系统设定 + 用户输入 → 多轮追问」。

In [ ]:
from langchain.chat_models import init_chat_model
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)

if READY:
    messages = [
        # 系统消息：设定人设
        ("system", "你是一个简洁的编程助手，每次回答不超过 50 字。"),
        # 用户消息
        ("user", "什么是函数调用？"),
    ]

    response = llm.invoke(messages)
    print("AI：", response.content)

    # ---------- 多轮对话：把 AI 的回复也放进列表 ----------
    messages.append(response)          # 记住 AI 刚说了什么
    messages.append(("user", "展开讲讲"))  # 用户追问
    response = llm.invoke(messages)
    print("AI（追问）：", response.content)

    # ---------- 工具消息示例 ----------
    # 调用工具后，要把 ToolMessage（工具结果）放进列表回传给模型：
    # from langchain_core.messages import ToolMessage
    # messages.append(ToolMessage(content="查询结果：上海 25 度", tool_call_id="call_xxx"))

### 预期输出

> ⚠️ 两行回答的正文是模型这次生成的，每次运行都不同；稳定的是「两轮都返回一行 `AI：` 前缀」。

```text
AI： 函数调用是执行函数代码，传入参数并获取返回值的过程。
AI（追问）： 函数调用：压栈保存现场，传参，跳转执行函数体；返回时弹栈恢复，取返回值继续。
```

### 3.2 完整版：四种消息「体检」

完整版先用一个 `dump` 函数把每条消息的「身份信息」打印出来，方便对照上面的四行表。

In [ ]:
# 新版导入路径（课案原文用的就是这条）；等价于 from langchain_core.messages import ...
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.chat_models import init_chat_model
from config import settings

# 课案写的是 ChatOpenAI(model=..., api_key=..., base_url=...)；
# 本项目统一走 init_chat_model，参数同样全部来自 settings。
llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


# ---------- 1. 消息对象体检：看清四种消息各自长什么样 ----------
def dump(tag: str, msg) -> None:
    """打印一条消息的「身份信息」，方便对照上面的表格。"""
    print(f"  {tag:<10} 类={type(msg).__name__:<14} type={msg.type:<7} content={msg.content!r}")
    # ToolMessage 才有 tool_call_id；AIMessage 要调工具时才有 tool_calls
    if isinstance(msg, ToolMessage):
        print(f"  {'':<10} tool_call_id={msg.tool_call_id}")
    if isinstance(msg, AIMessage) and msg.tool_calls:
        print(f"  {'':<10} tool_calls={msg.tool_calls}")

### 3.3 课案原文 + 多轮对话 + ToolMessage 结构

依次演示：① 课案原文两条消息（System + Human）写诗 → ② 把 `AIMessage` 追加进列表
实现多轮 → ③ 手工构造一对 `AIMessage(带 tool_calls) + ToolMessage` 看清「一次工具调用」
在消息列表里长什么样。

> 真正由智能体自动产生工具调用的过程见 `02_智能体与工具.ipynb`；这里手工拼是为了看结构。

In [ ]:
if READY:
    # ---------- 2. 课案原文：SystemMessage + HumanMessage 写七言绝句 ----------
    messages = [
        SystemMessage("你是一位唐诗专家"),
        HumanMessage("写一首关于春天的七言绝句"),
    ]
    print("===== 2. 课案原文：两条消息 =====")
    # 打印的是「我们即将发出去的东西」，不是模型的返回 —— 对照上面那张四行表看：
    # 这里刚好是「system 在最前、human 随后」的合规顺序。
    for index, msg in enumerate(messages, start=1):
        dump(f"[{index}]", msg)

    # 为什么可以整个列表丢进去：模型这一层无状态，每次请求都要重新带上全部消息，
    # 所以「怎么拼这个列表」就是 LangChain 消息体系的全部工作量所在。
    response = llm.invoke(messages)
    print("\n模型返回：", response.content)
    dump("[模型返回]", response)          # 模型返回的是 AIMessage

    # ---------- 3. 多轮对话 = 不断 append 消息 ----------
    # 注意：不是「模型记住了」，而是「我们把上一轮它说的话又原样递回去了」。
    print("\n===== 3. 多轮对话：把 AIMessage 追加进列表 =====")
    messages.append(response)                          # 追加模型刚才的回复（AIMessage）
    messages.append(HumanMessage("把这首诗改成五言"))    # 追加新一轮用户输入（HumanMessage）
    print(f"  当前列表共 {len(messages)} 条消息，角色顺序：{[m.type for m in messages]}")

    response = llm.invoke(messages)
    print("模型返回：", response.content)

# ---------- 4. ToolMessage：工具结果的载体 ----------
# 只有在模型发起工具调用之后才会出现，必须与 tool_call_id 配对。
# 这里手工造一对「AIMessage(带 tool_calls) + ToolMessage」展示结构，
# 真正由智能体自动产生的过程见 03_智能体_jxsd.py / 04_工具_jxsd.py。
print("\n===== 4. ToolMessage 的结构（手工构造，仅示意） =====")
# 手工拼这两条消息，是为了让你看清楚「一次工具调用」在消息列表里长什么样：
# AIMessage 用 tool_calls[].id 提出问题，ToolMessage 用 tool_call_id 回答 —— 一问一答。
# 框架自动做这件事时，就是 create_agent 内部那个 tool 节点。
ai_with_tool_call = AIMessage(
    content="",                                   # 要调工具时，正文通常为空
    tool_calls=[
        {
            "name": "get_weather",                # 工具名
            "args": {"city": "北京"},              # 参数（由模型按 schema 填）
            "id": "call_1",                       # 本次调用的唯一编号
            "type": "tool_call",
        }
    ],
)
tool_message = ToolMessage(
    content="北京：晴，25℃",                        # 工具真正返回的内容
    name="get_weather",
    tool_call_id="call_1",                        # 必须等于上面那条 tool_call 的 id
)
dump("[AI]", ai_with_tool_call)
dump("[TOOL]", tool_message)

### 预期输出

> ⚠️ 「模型返回」的诗词正文是模型这次生成的，每次运行都不同；
> 稳定的是 `[1]`/`[2]`/`[AI]`/`[TOOL]` 的 dump 结构、以及「角色顺序」那一行。

```text
===== 2. 课案原文：两条消息 =====
  [1]        类=SystemMessage  type=system  content='你是一位唐诗专家'
  [2]        类=HumanMessage   type=human   content='写一首关于春天的七言绝句'

模型返回： 《七绝·春景》
东君送暖入芳丛，万树桃花映日红。
垂柳含烟摇碧水，春光醉客画图中。

===== 3. 多轮对话：把 AIMessage 追加进列表 =====
  当前列表共 4 条消息，角色顺序：['system', 'human', 'ai', 'human']
模型返回： 《五绝·春景》
东君送暖风，万树映春红。
柳烟摇碧水，春光醉画中。

===== 4. ToolMessage 的结构（手工构造，仅示意） =====
  [AI]       类=AIMessage      type=ai      content=''
             tool_calls=[{'name': 'get_weather', 'args': {'city': '北京'}, 'id': 'call_1', 'type': 'tool_call'}]
  [TOOL]     类=ToolMessage    type=tool    content='北京：晴，25℃'
             tool_call_id=call_1
```

重点看第 3 段：追加之后列表里是 `system, human, ai, human` —— 顺序铁律一目了然。

### 3.4 三种等价写法（语法糖对照）

三种写法 LangChain 内部都会归一化成消息对象：

| 写法 | 归一化结果 |
|---|---|
| `("user", "你好")` | `HumanMessage("你好")` |
| `{"role": "user", "content": "你好"}` | `HumanMessage("你好")` |
| `"你好"`（裸字符串） | `HumanMessage("你好")` ← **丢了角色语义，不推荐** |

下面只列前两种；第三种（裸字符串）刻意没放进来 —— 它就是那个「角色语义丢失」的反例，
单独跑一遍看不出差别，容易误导。

In [ ]:
if READY:
    # ---------- 5. 元组写法 / 裸字符串：语法糖对照 ----------
    # 三种写法等价，LangChain 内部都会归一化成消息对象：
    #   ("user", "你好")            → HumanMessage("你好")
    #   {"role": "user", "content": "你好"} → HumanMessage("你好")
    #   "你好"                       → HumanMessage("你好")   ← 丢了角色语义，不推荐
    print("\n===== 5. 三种等价写法 =====")
    # 下面两行只列「元组」和「字典」两种，第三种（裸字符串）刻意没放进来 ——
    # 它就是上面注释里那个「角色语义丢失」的反例，单独跑一遍看不出差别，容易误导。
    sugar_forms = [
        ("元组写法", [("system", "只回答一个字"), ("user", "中国的首都是？")]),
        ("字典写法", [{"role": "system", "content": "只回答一个字"}, {"role": "user", "content": "中国的首都是？"}]),
    ]
    for tag, msgs in sugar_forms:
        # 归一化发生在 invoke 内部：LangChain 把元组/字典都转成 Message 对象再发给接口，
        # 所以两种写法对模型而言完全等价（差别只在 Python 侧的写法习惯）。
        normalized = llm.invoke(msgs)
        print(f"  {tag}：{normalized.content}")

### 预期输出

> ⚠️ 「京」这个答案本身是模型生成的，每次运行都不同；稳定的是「两种写法都走 `llm.invoke`」这件事。

```text
===== 5. 三种等价写法 =====
  元组写法：京
  字典写法：京
```

## 4. 结构化输出：让模型按 Schema 吐 JSON

前两节的 `invoke` 返回的是**自由文本**，下游程序得自己抠字符串。
结构化输出用 **Pydantic 模型** 定义输出结构，底层靠 **Function Calling**：
框架把 Pydantic 转成 JSON Schema，再让模型「只能」按这份 schema 输出参数，
LangChain 解析校验后直接还你一个 Pydantic 实例。

两种写法（它们用同一套底层机制，都依赖端点的结构化输出能力）：

| 写法 | 谁来做结构化 | 适合场景 |
|---|---|---|
| `llm.with_structured_output(模型)` | 裸模型 | 纯信息抽取、分类，不需要工具 |
| `create_agent(..., response_format=模型)` | 智能体 | 需要「先查资料/调工具，再产出结构化结果」 |

结果都在「结构化对象」里：智能体放 `result["structured_response"]`，裸模型直接把对象返回给你。

### 4.1 为什么本机跑不通（降级说明）

结构化输出要靠**端点能力**：要么支持原生 `json_schema`，要么支持强制 `tool_choice`
（function calling）。本机 `.env` 指向 DeepSeek 官方端点（`deepseek-flash`，
**思考模型**），两条路都会被 400 拒绝：

- 原生 `json_schema` → `400 This response_format type is unavailable now`
- 强制 `tool_choice` → `400 Thinking mode does not support this tool_choice`
- `method="json_mode"` 能通 HTTP 但**不约束字段名**（实测把 `goal/steps` 输出成 `目标/步骤`）

源文件因此内置了「打印中文提示并跳过」的降级分支：**不静默失败、也不让课案崩掉**。
把 `.env` 的 `API_KEY` / `BASE_URL` / `MODEL_NAME` 切回支持结构化输出的端点即可跑通。

### 4.2 课案原版：`with_structured_output`

定义两个 Pydantic 输出结构 —— `MovieReview`（信息抽取）和 `RouteDecision`（意图路由）。

In [ ]:
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


# ---------- 1. 定义输出结构 ----------
class MovieReview(BaseModel):
    """影评结构"""
    title: str = Field(description="电影名称")
    score: int = Field(description="评分 1-10")
    reason: str = Field(description="一句话评价理由")
    tags: list[str] = Field(description="类型标签，如['科幻','剧情']")


class RouteDecision(BaseModel):
    """路由决策结构"""
    intent: str = Field(description="用户意图：chat / order / complaint")


# 说明：结构化输出要靠**端点能力** —— 要么支持原生 json_schema，要么支持强制
# tool_choice（function calling）。本机 .env 若指向 DeepSeek 官方端点
# （deepseek-flash / deepseek-v4-pro 都是思考模型），两条路都会被 400 拒绝：
#   · json_schema → "This response_format type is unavailable now"
#   · 强制工具选择 → "Thinking mode does not support this tool_choice"
# 所以这里兜一层中文提示：不静默失败、也不让课案崩掉。把 .env 的
# API_KEY / BASE_URL / MODEL_NAME 切回支持结构化输出的端点即可跑通。
# 三组端点的实测对照见 `20_上下文工程_官方补充.py` 文末「实测结论」第 5 条。
try:
    # ---------- 2. 信息抽取 ----------
    reviewer = llm.with_structured_output(MovieReview)
    review = reviewer.invoke("评价一下《流浪地球2》：视觉震撼，剧情稍散，8分吧")
    print("结构化结果：")
    print("  片名：", review.title)
    print("  评分：", review.score)
    print("  理由：", review.reason)
    print("  标签：", review.tags)

    # ---------- 3. 意图路由（客服分流常用） ----------
    router = llm.with_structured_output(RouteDecision)
    decision = router.invoke("我要投诉你们物流太慢了！")
    print("路由决策：", decision.intent)
except Exception as exc:
    # 注意：这里只兜「端点不支持」这一类，其余异常同样会被打印出来看清原因
    print(f"[跳过] 本端点跑不通结构化输出：{type(exc).__name__}")
    print(f"       {str(exc)[:200]}")
    print("       → 换一个支持 json_schema 或 function calling 的端点即可。")

### 预期输出（本机）

```text
[跳过] 本端点跑不通结构化输出：OpenAIInvalidRequestError
       Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}
       → 换一个支持 json_schema 或 function calling 的端点即可。
```

换到支持 `json_schema` / function calling 的端点后，同一格会变成：

```text
结构化结果：
  片名： 流浪地球2
  评分： 8
  理由： 视觉震撼，剧情稍散
  标签： ['科幻', '剧情']
路由决策： complaint
```

### 4.3 完整版：`create_agent(response_format=...)`

完整版讲智能体版的结构化输出，并演示更复杂结构（嵌套列表）+「先调工具再结构化」。
课案原文那句「需要使用没有思考模式的模型」的注释，正是本节最关键的坑。

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
)


# ---------- 1. 课案原文的输出模型 ----------
class ContactInfo(BaseModel):
    """个人联系信息"""
    # Field(description=...) 会原样进入 JSON Schema 的 description，
    # 模型就是靠这几句话知道「哪个字段该填什么」——所以描述要写清楚。
    name: str = Field(description="姓名")
    email: str = Field(description="邮箱")
    phone: str = Field(description="电话")


# ---------- 2. 更复杂一点：嵌套列表 + 枚举 ----------
class Ticket(BaseModel):
    """一张待办工单"""
    title: str = Field(description="工单标题，不超过 10 个字")
    priority: str = Field(description="优先级，只能是 高/中/低 三者之一")
    tags: list[str] = Field(description="标签，如 ['后端', '线上']")


class TicketList(BaseModel):
    """工单列表（演示嵌套结构）"""
    items: list[Ticket] = Field(description="拆解出来的工单列表")


# ---------- 3. 带工具的智能体 + 结构化输出：先查再答 ----------
@tool
def lookup_user(user_id: str) -> str:
    """按用户编号查询用户资料。user_id：用户编号，如 u1"""
    # 演示用：真实项目里这里会查数据库
    return "u1：李四，lisi@example.com，13900139000"


def run_case(tag: str, agent, question: str) -> None:
    """统一的调用 + 取结果 + 异常兜底，保证脚本不会因为模型不支持而崩掉。"""
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": question}]})
        print(f"===== {tag} =====")
        print("structured_response：", result["structured_response"])
        print("类型：", type(result["structured_response"]).__name__)
        # 结构化结果的本质：可以直接按属性取值，不再需要正则去抠字符串
        print("模型 dump 出来的字典：", result["structured_response"].model_dump())
    except Exception as exc:
        # 最常见的失败原因就是课案注释里说的「模型带思考模式，不支持强制 tool_choice」
        print(f"[跳过] {tag} 结构化输出失败：{type(exc).__name__}")
        print("       若报错与 tool_choice / thinking 有关，请把 settings.model_name")
        print("       换成没有思考模式的对话模型（课案用的是 deepseek-chat）。")
        print("       详情：", str(exc)[:200])

### 4.4 四步演示（本机全部命中降级）

依次跑：① 课案原文抽取联系人 → ② 嵌套结构拆工单 → ③ 先调工具再结构化 →
④ 对照写法 `with_structured_output`。本机（思考模型）每一步都会打印 `[跳过]`。

In [ ]:
# ---------- 4. 课案原文：最简结构化输出 ----------
# response_format 传 Pydantic 类，智能体就会把最终答案「塞」成这个类的实例，
# 存放在 result["structured_response"] 里（messages 里仍有一次普通回复）。
contact_agent = create_agent(model=llm, response_format=ContactInfo)
run_case("4. 课案原文：抽取联系人信息", contact_agent, "李四，lisi@example.com，13900139000")

# ---------- 5. 课案原样的嵌套结构 ----------
print()
ticket_agent = create_agent(model=llm, response_format=TicketList)
run_case(
    "5. 嵌套结构：把需求拆成工单",
    ticket_agent,
    "登录页报 500 要马上修；顺便把文档补一下，不急。",
)

# ---------- 6. 智能体 + 工具 + 结构化输出 ----------
# 这是 create_agent 版比 with_structured_output 强的地方：
# 模型可以先调 lookup_user 拿资料，再把结果整理成 ContactInfo。
print()
tool_agent = create_agent(model=llm, tools=[lookup_user], response_format=ContactInfo)
run_case("6. 先查工具再结构化", tool_agent, "帮我查一下 u1 这个用户的联系方式")

# ---------- 7. 对照：裸模型的 with_structured_output ----------
# 不经过智能体，少一次图调度，速度更快；但没法「先调工具再输出」。
# 两种写法用的是同一套底层机制（都是把 Pydantic 转 JSON Schema + 强制 tool_choice），
# 所以第 6 步报错时这一步大概率也报错 —— 排查方向是同一个。
print("\n===== 7. 对照写法：llm.with_structured_output =====")
try:
    reviewer = llm.with_structured_output(ContactInfo)
    info = reviewer.invoke("李四，lisi@example.com，13900139000")
    print("直接拿到对象：", info)
    print("直接取属性：name =", info.name)   # 不再需要 json.loads / 正则
except Exception as exc:
    print("[跳过] with_structured_output 失败：", type(exc).__name__, str(exc)[:200])

### 预期输出（本机）

```text
[跳过] 4. 课案原文：抽取联系人信息 结构化输出失败：OpenAIInvalidRequestError
       若报错与 tool_choice / thinking 有关，请把 settings.model_name
       换成没有思考模式的对话模型（课案用的是 deepseek-chat）。
       详情： Error code: 400 - {'error': {'message': 'Thinking mode does not support this tool_choice', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

[跳过] 5. 嵌套结构：把需求拆成工单 结构化输出失败：OpenAIInvalidRequestError
       若报错与 tool_choice / thinking 有关，请把 settings.model_name
       换成没有思考模式的对话模型（课案用的是 deepseek-chat）。
       详情： Error code: 400 - {'error': {'message': 'Thinking mode does not support this tool_choice', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

[跳过] 6. 先查工具再结构化 结构化输出失败：OpenAIInvalidRequestError
       若报错与 tool_choice / thinking 有关，请把 settings.model_name
       换成没有思考模式的对话模型（课案用的是 deepseek-chat）。
       详情： Error code: 400 - {'error': {'message': 'Thinking mode does not support this tool_choice', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}

===== 7. 对照写法：llm.with_structured_output =====
[跳过] with_structured_output 失败： OpenAIInvalidRequestError Error code: 400 - {'error': {'message': 'This response_format type is unavailable now', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_request_error'}}
```

换到非思考模型后，第 4 步会打印 `name='李四' email='lisi@example.com' phone='13900139000'`，
第 6 步会先 `lookup_user` 查资料再整理成 `ContactInfo` —— 这正是智能体版比裸模型版强的地方。

## 5. 官方文档补充：模型配置进阶

课案 `01_模型` 讲了「怎么把模型接上」；这一篇（对照官方 `models.mdx`）讲的是
**接上之后的工程细节**：

- 模型参数：`temperature` / `max_tokens` / `timeout` / `max_retries` 各自管什么；
- 连接韧性：重试策略（官方默认 `max_retries=6`、指数退避 + 抖动，只重试网络错误 / 429 / 5xx，401 与 404 不重试）；
- 限流：`InMemoryRateLimiter` 挂在模型上，避免把自己的额度打穿；
- 用量核算：`usage_metadata` 的完整字段（含推理 token 与缓存命中）；
- 内容块与多模态：`content_blocks` 的形态，以及本机网关的现实支持情况。

⚠️ 本节有两个**实测得出的重要结论**（官方文档不会告诉你，因为是本机网关的脾气）：

A. **中转网关可能无视 `max_tokens`**：本机设 `max_tokens=16` 让它写长文，实际返回数千输出 token ——
   参数被接受但没被执行，控长度要提示词 + 自截断 + 监控 `finish_reason`；
B. **图片内容块在本机不稳定**：带 image 块的请求有时报 `OpenAIConnectionError`（纯文本正常），
   有时又被网关接受 —— 多模态缺口维持「待补（要支持视觉的端点）」。

### 5.1 公共设施：`build_model` / `collect_usage` / 用量累加器

`build_model` 按 `.env` 配置造模型实例并允许覆盖参数（演示里关掉自动重试避免等满退避）；
`collect_usage` 把每次响应的 `usage_metadata` 累加进 `USAGE_TOTAL`。

In [ ]:
import time

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.rate_limiters import InMemoryRateLimiter

from config import settings


def build_model(**overrides):
    """按 .env 里的配置造一个模型实例，允许覆盖任意模型参数。"""
    kwargs = {
        "model_provider": "openai",
        "model": settings.model_name,
        "api_key": settings.api_key,
        "base_url": settings.base_url,
        "max_retries": 0,      # 演示里关掉自动重试，避免每次都等 6 轮退避
    }
    kwargs.update(overrides)
    return init_chat_model(**kwargs)


# 用量累加器：核算成本的基础（官方字段见 Demo 5）
USAGE_TOTAL = {"input_tokens": 0, "output_tokens": 0, "reasoning": 0, "cache_read": 0}


def collect_usage(response) -> dict:
    """把一次响应的 usage_metadata 累加进来，并返回本次明细。"""
    usage = getattr(response, "usage_metadata", None) or {}
    details_out = usage.get("output_token_details") or {}
    details_in = usage.get("input_token_details") or {}
    USAGE_TOTAL["input_tokens"] += usage.get("input_tokens", 0)
    USAGE_TOTAL["output_tokens"] += usage.get("output_tokens", 0)
    USAGE_TOTAL["reasoning"] += details_out.get("reasoning", 0)
    USAGE_TOTAL["cache_read"] += details_in.get("cache_read", 0)
    return usage

### 5.2 Demo 1：模型参数与响应元数据

官方参数表：`temperature`（随机性）、`max_tokens`（输出上限）、`timeout`（秒）、
`max_retries`（默认 6），通过 `init_chat_model` 的 `**kwargs` 传入。
响应对象带着三样情报：`response_metadata`（模型名/结束原因/token 用量）、
`content_blocks`（内容块，LangChain 1.x 标准形态）、`usage_metadata`（成本核算依据）。

In [ ]:
def demo_1_parameters_and_metadata() -> None:
    print("=" * 70)
    print("Demo 1：模型参数与响应元数据")
    print("=" * 70)

    model = build_model(temperature=0.2, timeout=60)
    response = model.invoke("用一句话说明 temperature 参数的作用。")

    print(f"  回答：{str(response.content)[:90]}…")
    print(f"  content_blocks：{response.content_blocks}")
    metadata = response.response_metadata or {}
    print(f"  模型名：{metadata.get('model_name')}｜结束原因：{metadata.get('finish_reason')}")
    print(f"  token 用量：{collect_usage(response)}")
    print(
        "  ↑ 三个值得记住的字段：\n"
        "    · response_metadata.finish_reason：**是不是被截断**看它（stop=正常结束）；\n"
        "    · content_blocks：LangChain 1.x 的标准内容形态（文本/推理/工具调用/图片…）；\n"
        "    · usage_metadata：含推理 token 与缓存命中，是成本核算的唯一依据。"
    )

### 5.3 Demo 2：`max_tokens` 的实测真相

官方文档说 `max_tokens` 限制输出长度；**本机网关实测无视它**（参数被接受但不生效）。

In [ ]:
def demo_2_max_tokens_reality() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：max_tokens 的实测真相 —— 中转网关可能无视它")
    print("=" * 70)

    model = build_model(max_tokens=16)
    try:
        # 只要 300 字就够证明问题：max_tokens=16 若真生效，输出会被截到十几个 token。
        # （早先要求"至少两千字"时更慢更费额度，结论完全一样，所以缩短。）
        response = model.invoke("用大约 300 字介绍 LangGraph 的持久化机制。")
    except Exception as exc:  # noqa: BLE001
        print(f"  本次调用失败（网关抖动/额度问题，非代码问题）：{type(exc).__name__}: {str(exc)[:100]}")
        print("  重跑一次通常即可；本 Demo 要演示的是参数是否被执行，换个时段再验也行。")
        return
    usage = collect_usage(response)
    text = str(response.content)
    print(f"  请求参数 max_tokens=16，实际返回：{len(text)} 字符 / {usage.get('output_tokens')} 输出 token")
    print(f"  finish_reason：{(response.response_metadata or {}).get('finish_reason')}")
    if usage.get("output_tokens", 0) > 100:
        print(
            "  ⚠️ 结论：**本机网关没有执行 max_tokens**（参数被接受但不生效）——\n"
            "     换端点/直连官方 API 时通常有效，但**永远不要假设它一定生效**。\n"
            "     要控长度就三管齐下：提示词里限字数 + 自己截断 + 用 finish_reason 监控。"
        )
    else:
        print("  ✔ 本次 max_tokens 生效（输出被压到阈值内）")

### 5.4 Demo 3：连接韧性 —— `timeout` 与 `max_retries`

官方语义：`max_retries` 默认 6，指数退避 + 抖动；只对网络错误、429、5xx 重试，
401/404 不重试。本 Demo 用一个**必然超时**的小 `timeout` 证明超时参数真的传到了 HTTP 层。

In [ ]:
def demo_3_resilience() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：连接韧性 —— timeout 生效验证（故意造一次超时）")
    print("=" * 70)

    model = build_model(timeout=0.001)     # 1 毫秒：必然超时
    started = time.time()
    try:
        model.invoke("你好")
        print("  居然没超时？（不符合预期）")
    except Exception as exc:  # noqa: BLE001
        elapsed = time.time() - started
        print(f"  预期内失败：{type(exc).__name__}（耗时 {elapsed:.2f}s）")
        print(f"  异常文本：{str(exc)[:100]}")
    print(
        "  ↑ timeout 确实被传给了底层 HTTP 客户端（1 毫秒必然失败）。\n"
        "    生产建议：网络不稳就给 timeout=120、max_retries=10~15，并配 checkpointer；\n"
        "    注意**不是所有错误都重试**（401/404 直接失败，免得白等 6 轮）。"
    )

### 5.5 Demo 4：限流器（零模型调用）

官方把限流器挂在**模型实例**上（`init_chat_model(..., rate_limiter=limiter)`），
之后每次模型调用都先申请令牌。本 Demo 不调模型，直接测限流器本身 —— 结论干净、还不花钱。
参数：`requests_per_second` / `check_every_n_seconds` / `max_bucket_size`。

In [ ]:
def demo_4_rate_limiter() -> None:
    print("\n" + "=" * 70)
    print("Demo 4：限流器 —— 每秒 2 次，连要 4 个令牌要等多久？")
    print("=" * 70)

    limiter = InMemoryRateLimiter(
        requests_per_second=2,       # 每秒放行 2 次
        check_every_n_seconds=0.05,  # 检查间隔（越小越精确、越费 CPU）
        max_bucket_size=1,           # 桶容量：允许的突发量
    )
    started = time.time()
    stamps = []
    for index in range(4):
        limiter.acquire()            # 向限流器申请一次放行（同步版）
        stamps.append(time.time() - started)
        print(f"  第 {index + 1} 个令牌拿到，累计 {stamps[-1]:.2f}s")
    gaps = [round(stamps[i + 1] - stamps[i], 2) for i in range(len(stamps) - 1)]
    print(f"  相邻间隔：{gaps} 秒（2 req/s 的稳态间隔应约 0.5 秒）")
    print(
        "  ↑ 用法：`init_chat_model(..., rate_limiter=limiter)` —— 挂上之后，\n"
        "    agent 里每一次模型调用都会先申请令牌，天然把并发压在你允许的速率内。\n"
        "    `max_bucket_size` 是突发容量：设 1 表示不许攒额度、必须匀速。"
    )

### 5.6 Demo 5：token 用量核算（成本核算的基础）

跑两次调用，累加出 `input / output / reasoning / cache_read` 四个口径。

In [ ]:
def demo_5_token_accounting() -> None:
    print("\n" + "=" * 70)
    print("Demo 5：token 用量核算（成本核算的基础）")
    print("=" * 70)

    # 先清零：USAGE_TOTAL 是模块级累加器，前面几个 Demo 也往里加过
    # （不清零就会打印出「本次 2 次调用」却包含前面所有调用的总量 —— 本文件踩过）
    for key in USAGE_TOTAL:
        USAGE_TOTAL[key] = 0

    model = build_model()
    rounds = 0
    for question in ("一句话解释什么是向量检索。", "一句话解释什么是重排序。"):
        response = model.invoke(question)
        rounds += 1
        usage = collect_usage(response)
        print(f"  提问：{question}")
        print(f"    输入 {usage.get('input_tokens')} / 输出 {usage.get('output_tokens')} "
              f"/ 其中推理 {((usage.get('output_token_details') or {}).get('reasoning'))} "
              f"/ 缓存命中 {((usage.get('input_token_details') or {}).get('cache_read'))}")
    print(f"\n  本段累计（{rounds} 次调用）：{USAGE_TOTAL}")
    print(
        "  ↑ 注意两点（都是本机实测观察到的）：\n"
        "    ① **推理 token 计入输出**：本机模型输出里很大一部分是 reasoning，成本要算进去；\n"
        "    ② **缓存命中会显著省钱**：input_token_details.cache_read 命中越多、输入越便宜。\n"
        "    把 collect_usage 这种累加器挂在生产链路里，才能回答「这个功能一次多少钱」。"
    )

### 5.7 Demo 6：多模态（内容块）在本机的现实

内容块是 LangChain 1.x 的跨厂商标准（支持视觉的模型都能吃）。本机网关对图片块
**不稳定**（时而 `OpenAIConnectionError`、时而被接受），所以这段以 try/except 收尾。

In [ ]:
# 是否真的发一次图片块请求。默认开启 —— 实测它是**可捕获的异常**，不会带崩脚本：
#     图片块调用失败：OpenAIConnectionError: Connection error.（纯文本调用同一端点正常）
MULTIMODAL_LIVE_TEST = True


def demo_6_multimodal_reality() -> None:
    print("\n" + "=" * 70)
    print("Demo 6：多模态（内容块）在本机的现实")
    print("=" * 70)

    # 一个 1x1 的透明 PNG（base64），用来最小化测试图片输入
    tiny_png = (
        "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42mNkYPhfDwAChwGA60e6kgAAAABJRU5ErkJggg=="
    )
    message = HumanMessage(content=[
        {"type": "text", "text": "这张图片是什么颜色？一句话回答。"},
        {"type": "image", "base64": tiny_png, "mime_type": "image/png"},
    ])
    print("  内容块的标准写法（跨厂商通用，支持视觉的模型都能吃）：")
    print(f"    HumanMessage(content=[{{'type': 'text', ...}}, "
          f"{{'type': 'image', 'base64': '<base64>', 'mime_type': 'image/png'}}])")

    if not MULTIMODAL_LIVE_TEST:
        print("\n  （已跳过真实调用：MULTIMODAL_LIVE_TEST=False）")
        print("  实测结论：本机网关对图片块**不可用** —— OpenAIConnectionError: Connection error，")
        print("    而同一端点的纯文本调用正常。")
    else:
        model = build_model()
        try:
            response = model.invoke([message])
            print(f"\n  ✔ 网关接受了图片块：{str(response.content)[:80]}")
        except Exception as exc:  # noqa: BLE001
            print(f"\n  ✘ 图片块调用失败：{type(exc).__name__}: {str(exc)[:120]}")
            print("    （这是**可捕获的异常**，脚本会继续往下跑，不影响其它 Demo）")

    print(
        "  ↑ 结论：多模态相关缺口在本仓库继续挂「待补（要支持视觉的端点）」。\n"
        "    代码写法本身是对的（内容块是 LangChain 1.x 的跨厂商标准），\n"
        "    换一个支持视觉的模型端点即可跑通，无需改代码。"
    )

### 5.8 执行全部 Demo

六个 Demo 依次跑。Demo 4 零模型调用，其余依赖模型 —— 全部包在 `if READY:` 里。

In [ ]:
if READY:
    demo_1_parameters_and_metadata()
    demo_2_max_tokens_reality()
    demo_3_resilience()
    demo_4_rate_limiter()
    demo_5_token_accounting()
    demo_6_multimodal_reality()
    print("\n全部 Demo 执行完毕。")

### 预期输出

> ⚠️ 这一段几乎每行都带易变内容：模型回答的措辞、`max_tokens` 实测的字符/token 数、
> Demo 3 的耗时、Demo 4 的累计时间戳、Demo 5 的 token 数、Demo 6 是「接受还是报错」，
> 每次运行都不同。稳定的是每段的结构与「⚠️ 本机网关没有执行 max_tokens」这个结论。

```text
======================================================================
Demo 1：模型参数与响应元数据
======================================================================
  回答：temperature 参数用于调节模型生成时的随机性，值越高输出越多样有创意，值越低输出越确定保守。…
  content_blocks：[{'type': 'text', 'text': 'temperature 参数用于调节模型生成时的随机性，值越高输出越多样有创意，值越低输出越确定保守。'}]
  模型名：deepseek-flash｜结束原因：stop
  token 用量：{'input_tokens': 38, 'output_tokens': 113, 'total_tokens': 151, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 86}}
  ↑ 三个值得记住的字段：
    · response_metadata.finish_reason：**是不是被截断**看它（stop=正常结束）；
    · content_blocks：LangChain 1.x 的标准内容形态（文本/推理/工具调用/图片…）；
    · usage_metadata：含推理 token 与缓存命中，是成本核算的唯一依据。

======================================================================
Demo 2：max_tokens 的实测真相 —— 中转网关可能无视它
======================================================================
  请求参数 max_tokens=16，实际返回：480 字符 / 666 输出 token
  finish_reason：stop
  ⚠️ 结论：**本机网关没有执行 max_tokens**（参数被接受但不生效）——
     换端点/直连官方 API 时通常有效，但**永远不要假设它一定生效**。
     要控长度就三管齐下：提示词里限字数 + 自己截断 + 用 finish_reason 监控。

======================================================================
Demo 3：连接韧性 —— timeout 生效验证（故意造一次超时）
======================================================================
  预期内失败：OpenAITimeoutError（耗时 0.06s）
  异常文本：Request timed out.
  ↑ timeout 确实被传给了底层 HTTP 客户端（1 毫秒必然失败）。

======================================================================
Demo 4：限流器 —— 每秒 2 次，连要 4 个令牌要等多久？
======================================================================
  第 1 个令牌拿到，累计 0.50s
  第 2 个令牌拿到，累计 1.01s
  第 3 个令牌拿到，累计 1.51s
  第 4 个令牌拿到，累计 2.01s
  相邻间隔：[0.5, 0.5, 0.5] 秒（2 req/s 的稳态间隔应约 0.5 秒）

======================================================================
Demo 5：token 用量核算（成本核算的基础）
======================================================================
  提问：一句话解释什么是向量检索。
    输入 36 / 输出 88 / 其中推理 55 / 缓存命中 0
  提问：一句话解释什么是重排序。
    输入 36 / 输出 149 / 其中推理 117 / 缓存命中 0

  本段累计（2 次调用）：{'input_tokens': 72, 'output_tokens': 237, 'reasoning': 172, 'cache_read': 0}

======================================================================
Demo 6：多模态（内容块）在本机的现实
======================================================================
  内容块的标准写法（跨厂商通用，支持视觉的模型都能吃）：
    HumanMessage(content=[{'type': 'text', ...}, {'type': 'image', 'base64': '<base64>', 'mime_type': 'image/png'}])

  ✔ 网关接受了图片块：这张图片是浅紫色。

全部 Demo 执行完毕。
```

## 小结

- **模型**：`init_chat_model` 是工厂，`ChatOpenAI` 是车间 —— 运行时同一对象，差别只在可切换性；
- **消息**：4 种角色（`system` / `human` / `ai` / `tool`），顺序铁律 + `tool_call_id` 配对；
  「记忆」= 不断 append 消息再整列重喂；
- **结构化输出**：Pydantic 转 JSON Schema + Function Calling；两种写法
  （裸模型 `with_structured_output` vs 智能体 `create_agent(response_format=...)`）
  走同一套底层，都依赖端点能力（本机思考模型不支持，走降级）；
- **配置进阶**：`temperature` / `timeout` / `max_retries`（默认 6、只重试网络/429/5xx）、
  限流器挂模型实例、`usage_metadata` 含推理 token 与缓存命中。

下一课 `02_智能体与工具.ipynb` 会看到：本节手工摆出来的 `ToolMessage` 是怎么被
`create_agent` 的 tools 节点自动填进消息列表的。

## 常见坑

1. **结构化输出必须用「没有思考模式」的模型**：本机 `deepseek-flash` 是思考模型，
   原生 `json_schema` 报 `This response_format type is unavailable now`、
   强制 `tool_choice` 报 `Thinking mode does not support this tool_choice`；
   换非思考模型或支持 function calling 的端点即可跑通。
2. **别假设 `max_tokens` 生效**：中转网关可能忽略它（本机实测忽略），
   控长度要靠提示词 + 自截断 + 监控 `finish_reason`。
3. **推理 token 计入输出**：本机模型输出里 reasoning 占大头，估算成本漏了它会低估。
4. **`max_retries` 默认 6**：调试/演示务必显式设 0，否则一次失败要等满退避序列；
   生产里反而要调大它 + 配 checkpointer。
5. **限流器挂在模型实例上，不是 agent 上**：同一实例被 agent 与工具共用时统一生效；
  换实例就换了一份额度；`max_bucket_size` 设 1 = 严格匀速。
6. **`ToolMessage` 必须紧跟发起 `tool_call` 的 `AIMessage` 且 `tool_call_id` 对得上**，
   否则供应商接口直接 400。
7. **别把裸字符串混进消息列表**：LangChain 会当成 `HumanMessage`，能跑但丢了角色语义。

## 官方链接

- 模型（Models）：<https://docs.langchain.com/oss/python/langchain/models>
- 消息（Messages）：<https://docs.langchain.com/oss/python/langchain/messages>
- 结构化输出（Structured Output）：<https://docs.langchain.com/oss/python/langchain/structured-output>
- 智能体（Agents）：<https://docs.langchain.com/oss/python/langchain/agents>
- 限流器（Rate limiters）：<https://docs.langchain.com/oss/python/langchain/models#rate-limiting>